#### Import Libraries and Files

In [ ]:
#Get python libraries for data manipulation
import pandas as pd
import numpy as np
from itertools import cycle
from datetime import timedelta, datetime as dt
import unicodedata #Used to normalize text and remove accents

import warnings #To ignore openpyxl warnings
warnings.filterwarnings("ignore", category=UserWarning, module="openpyxl") #Suppress openpyxl warnings

#import access.ipynb. This archive needs to be created manually and consider variables to do queries and connect to other apps.
%run ../../access.ipynb

#import 00_input_download.ipynb. This deletes old files and downloads all archives sent by distis saving them in input folder
%run "00_input_download.ipynb"

In [ ]:
#Import Ingram sales and stock files
#input_ingram_sales = pd.read_excel(r'../input/ingram_consolidado.xlsx', dtype=str)
input_ingram_sales = pd.read_excel(r'../input/ingram_sales_report.xlsx', sheet_name= 'ASUS_RET SO', dtype=str)
input_ingram_stock = pd.read_excel(r'../input/ingram_stock_report.xlsx', sheet_name= 'ASUS_RET INV', dtype=str)

#Import Nexsys sales and stock files
#input_nexsys_sales = pd.read_excel(r'../input/nexsys_consolidado.xlsx',sheet_name='Sales', dtype=str)
input_nexsys_sales = pd.read_excel(r'../input/nexsys_report.xlsx',sheet_name='Sales', dtype=str)
input_nexsys_stock = pd.read_excel(r'../input/nexsys_report.xlsx',sheet_name='Inventory', dtype=str)

#Get Tecnoglobal reports
input_tecnoglobal_sales = pd.read_excel(r'../input/tecnoglobal_report.xlsx', sheet_name='DatosPOS', header=0)
input_tecnoglobal_stock = pd.read_excel(r'../input/tecnoglobal_report.xlsx', sheet_name='DatosINV', header=0)

In [ ]:
#Import Intcomex sales and stock files. 
input_intcomex_sales = pd.read_excel(r'../input/intcomex_sales_report.xlsx', dtype=str)

# Be careful with Stock Excel file, headers may vary its position.
# Load the first few rows to detect the header row and detect the row that contains the header
df_temp = pd.read_excel(r'../input/intcomex_stock_report.xlsx', nrows=10)
header_row = df_temp[df_temp.iloc[:, 0].str.contains('AFF', na=False)].index[0]+1

# Load the full file using the detected header row
input_intcomex_stock = pd.read_excel(r'../input/intcomex_stock_report.xlsx', header=header_row, dtype=str)

In [ ]:
#Import PCFactory sales and stock files. 

# Be careful with Sales and Stock Excel file, headers may vary its position.
# Load the first few rows to detect the header row and detect the row that contains the header
df_temp_sales = pd.read_excel(r'../input/pcfactory_sales_report.xlsx', nrows=10)
df_temp_sales = df_temp_sales.replace(r'^\s*$', np.nan, regex=True)
header_row_sales = df_temp_sales[df_temp_sales.astype(str).apply(lambda x: x.str.contains('Distributor', case=False, na=False)).any(axis=1)].index[0] + 1
input_pcfactory_sales = pd.read_excel(r'../input/pcfactory_sales_report.xlsx', header=header_row_sales, dtype=str)

df_temp_stock = pd.read_excel(r'../input/pcfactory_stock_report.xlsx', nrows=10)
df_temp_stock = df_temp_stock.replace(r'^\s*$', np.nan, regex=True)
header_row_stock= df_temp_stock[df_temp_stock.astype(str).apply(lambda x: x.str.contains('Distributor', case=False, na=False)).any(axis=1)].index[0] + 1

input_pcfactory_stock = pd.read_excel(r'../input/pcfactory_stock_report.xlsx', header=header_row_stock, dtype=str)

#### General DF Format

In [ ]:
# Define columns for data manipulation
columns_to_maintain_sales = ['date_sales', 'sku', 'invoice', 'sell_out_units', 'currency_code',
                             'sell_out_value', 'reseller_code', 'reseller_origin', 'part_number']

columns_to_maintain_stock = ['sku', 'stock_units', 'stock_value', 'currency_code', 'part_number']

Format Ingram dfs by column name

In [ ]:
#Each df must have the following columns:
#disti_sales: country_id, importer_name, date_sales, part_number, sku, invoice, reseller_code, sell_out_units, sell_out_value
#disti_stock: country_id, importer_name, date_stock, stock_units, stock_costs, part_number

# Convert all columns to lowercase, replace spaces with underscores and strip leading/trailing spaces.
input_ingram_sales.columns = input_ingram_sales.columns.str.strip().str.lower().str.replace(" ", "_")
input_ingram_stock.columns = input_ingram_stock.columns.str.strip().str.lower().str.replace(" ", "_")

#Define columns to rename
columns_to_rename_ingram_sales = {
                    'cust_name': 'reseller_origin', #resellers name for Ingram
                    'sku': 'sku', #SKU for Ingram distributor
                    "manufacturer'spartnbr": 'part_number', #Part number from ASUS
                    'invoice_date' : 'date_sales', #Date sales reported by Ingram
                    'invoice_nbr': 'invoice', #Invoice number
                    'quantity_shipped': 'sell_out_units', #Units sold
                    'sale_(usd)': 'sell_out_value',#Sales in USD 
                    'taxexemptid': 'reseller_code' #Resellers code for Ingram 
}

columns_to_rename_ingram_stock = {
    'sku': 'sku', #SKU for Ingram distributor
    'boh': 'stock_units', #Stock in units
    'vdr_partnb': 'part_number' #part_number from ASUS
}

ingram_sales_renamed = input_ingram_sales.rename(columns = columns_to_rename_ingram_sales)
ingram_stock_renamed = input_ingram_stock.rename(columns = columns_to_rename_ingram_stock)

#set [currency_code] = USD
ingram_sales_renamed['currency_code'] = 'USD'
ingram_stock_renamed['currency_code'] = 'USD'

#Step 2: set columns in dataframe and add country_id, importer_name to both dataframes and stock_costs to stock dataframe
ingram_stock_renamed['stock_value'] = 0

ingram_sales_renamed = ingram_sales_renamed[columns_to_maintain_sales]
ingram_stock_renamed = ingram_stock_renamed[columns_to_maintain_stock]

#country_id = 1 is Chile code, 2 is Peru code
ingram_sales_renamed['country_id'] = 1
ingram_stock_renamed['country_id'] = 1

#For Chile: importer_id = 5 for Intcomex, 13 for Ingram, 22 for Nexsys  
ingram_sales_renamed['importer_id'] = 13
ingram_stock_renamed['importer_id'] = 13

#Format reported is '%d/%m/%Y'. Format used is '%Y-%m-%d'
ingram_sales_renamed['date_sales'] = pd.to_datetime(ingram_sales_renamed['date_sales'], format='%d/%m/%Y', errors='coerce')
ingram_sales_renamed['date_sales'] = ingram_sales_renamed['date_sales'].dt.strftime('%Y-%m-%d')

#Create and format date_stock to yyyy-mm-dd according to last week sunday for stock dataframe
day_sunday = pd.to_datetime(ingram_sales_renamed['date_sales'].iloc[0]) #Use the first row date_sales
end_week = day_sunday + timedelta(days=(6 - day_sunday.weekday()))
end_week_formatted = end_week.strftime('%Y-%m-%d')

#Create date_stock column
ingram_stock_renamed['date_stock'] = end_week_formatted


In [ ]:
# Map all reseller_origin values with Ingram national ID and conduct them to ASUS Eshop or marketplace stores
origins_eshop = [
    "ASUS Falabella MKP F", "ASUS Falabella MKP B",
    "ASUS eShop F", "ASUS eShop B",
    "ASUS Mercado Libre F", "ASUS Mercado Libre B",
    "ASUS Paris MKP F", "ASUS Paris MKP B",
    "ASUS Ripley MKP B"
]

# Step 2: Create a boolean mask to identify rows that should be corrected.
# The mask selects rows that:
#  - Currently have Ingram's RUT ("78137000-2")
#  - Have a 'reseller_origin' that matches one of the Eshop or marketplace origins
#  - Do not contain the word "INGRAM" (to exclude real Ingram records)

mask = ((ingram_sales_renamed["reseller_code"] == "78137000-2") &
        (ingram_sales_renamed["reseller_origin"].isin(origins_eshop)) &
        (~ingram_sales_renamed["reseller_origin"].str.contains("INGRAM", case=False, na=False)))

# Step 3: Apply the correction by updating the RUT for the selected rows.
ingram_sales_renamed.loc[mask, "reseller_code"] = "96516810-9"

Format Intcomex dfs by column name

In [ ]:
#Each df must have the following columns:
#disti_sales: country_id, importer_name, date_sales, part_number, sku, invoice, reseller_code, sell_out_units, sell_out_value
#disti_stock: country_id, importer_name, date_stock, stock_units, stock_costs

# Convert all columns to lowercase, replace spaces with underscores and strip leading/trailing spaces.
input_intcomex_sales.columns = input_intcomex_sales.columns.str.strip().str.lower().str.replace(" ", "_")
input_intcomex_stock.columns = input_intcomex_stock.columns.str.strip().str.lower().str.replace(" ", "_")

#Define columns to rename
columns_to_rename_intcomex_sales = {
    'trans_date': 'date_sales',
    'pack_transno': 'invoice',
    'customerid': 'reseller_code',
    'ship_qty': 'sell_out_units',
    'sku':'sku',
    'unit_initial_cost_us':'sales_cost',
    'mpn': 'part_number'
}

columns_to_rename_intcomex_stock = {
    'sku': 'sku',
    'oh':'stock_units',
    'initialcostus':'stock_value',
    'mpn': 'part_number'
}

#Rename columns in dataframe
intcomex_sales_renamed = input_intcomex_sales.rename(columns = columns_to_rename_intcomex_sales)
intcomex_stock_renamed = input_intcomex_stock.rename(columns = columns_to_rename_intcomex_stock)

#Step 2: to eliminate SUM of sell_out_units on final row intcomex_sales 
intcomex_sales_renamed.replace('', np.nan, inplace=True)
intcomex_sales_renamed = intcomex_sales_renamed.dropna(subset=['date_sales'])

#Create column [sell_out_value] and [sell_out_usd]
intcomex_sales_renamed['sell_out_value'] = intcomex_sales_renamed['sell_out_units'].astype(float) * intcomex_sales_renamed['sales_cost'].astype(float)
intcomex_sales_renamed['reseller_origin'] = None

#set [currency_code] = USD
intcomex_sales_renamed['currency_code'] = 'USD'
intcomex_stock_renamed['currency_code'] = 'USD'

# Set columns in dataframe and add country_id, importer_name to both dataframes and part_number to stock dataframe
intcomex_sales_renamed = intcomex_sales_renamed[columns_to_maintain_sales]
intcomex_stock_renamed = intcomex_stock_renamed[columns_to_maintain_stock]

#country_id = 1 is Chile code, 2 is Peru code
intcomex_sales_renamed['country_id'] = 1
intcomex_stock_renamed['country_id'] = 1

#For Chile: importer_id = 5 for Intcomex, 13 for Ingram, 22 for Nexsys  
intcomex_sales_renamed['importer_id'] = 5
intcomex_stock_renamed['importer_id'] = 5

#Format reported is '%m/%d/%Y'. Format used is '%Y-%m-%d

#Create and format date_stock to yyyy-mm-dd according to last week sunday for stock dataframe
day_sunday = pd.to_datetime(intcomex_sales_renamed['date_sales'].iloc[0]) #Use the first row date_sales
end_week = day_sunday + timedelta(days=(6 - day_sunday.weekday()))
end_week_formatted = end_week.strftime('%Y-%m-%d')

#Create date_stock column
intcomex_stock_renamed['date_stock'] = end_week_formatted

Format Nexsys dfs by column name

In [ ]:
#Each df must have the following columns:
#disti_sales: country_id, importer_name, date_sales, part_number, sku, invoice, reseller_code, reseller_name, sell_out_units, sell_out_value
#disti_stock: country_id, importer_name, date_stock, stock_units, stock_costs, part_number

# Convert all columns to lowercase, replace spaces with underscores and strip leading/trailing spaces.
input_nexsys_sales.columns = input_nexsys_sales.columns.str.strip().str.lower().str.replace(" ", "_")
input_nexsys_stock.columns = input_nexsys_stock.columns.str.strip().str.lower().str.replace(" ", "_")

#Define columns to rename
columns_to_rename_nexsys_sales = {
    'p#':'sku',
    'invoice': 'invoice',
    'tax-id':'reseller_code',
    'date':'date_sales',
    'qty':'sell_out_units',
    'revenue':'sell_out_value',
    'reseller': 'reseller_origin'
}

columns_to_rename_nexsys_stock = {
    'p#': 'sku',
    'on-hand': 'stock_units'
}

#Rename columns in dataframe
nexsys_sales_renamed = input_nexsys_sales.rename(columns= columns_to_rename_nexsys_sales)
nexsys_stock_renamed = input_nexsys_stock.rename(columns = columns_to_rename_nexsys_stock)

#Step 2: set columns in dataframe, add stock_costs and 'part_number to stock dataframe
nexsys_sales_renamed['part_number'] = nexsys_sales_renamed['sku']
nexsys_stock_renamed['part_number'] = nexsys_stock_renamed['sku']

#set [currency_code] = USD // [stock_usd] and [stock_value] = 0 (Nexsys does not provide this info)
nexsys_sales_renamed['currency_code'] = 'USD'
nexsys_stock_renamed['currency_code'] = 'USD'
nexsys_stock_renamed['stock_value'] = 0

# Keep only columns that are going to be used by 02_disti_sales_upload script
nexsys_sales_renamed = nexsys_sales_renamed[columns_to_maintain_sales]
nexsys_stock_renamed = nexsys_stock_renamed[columns_to_maintain_stock]

#country_id = 1 is Chile code, 2 is Peru code
nexsys_sales_renamed['country_id'] = 1
nexsys_stock_renamed['country_id'] = 1

#For Chile: importer_id = 5 for Intcomex, 13 for Ingram, 22 for Nexsys  
nexsys_sales_renamed['importer_id'] = 22
nexsys_stock_renamed['importer_id'] = 22

#Format reported is '%d/%m/%Y'. Format used is '%Y-%m-%d'
#nexsys_sales_renamed['date_sales'] = pd.to_datetime(nexsys_sales_renamed['date_sales'], format='%d-%m-%Y', errors='coerce', dayfirst=True)
#nexsys_sales_renamed['date_sales'] = nexsys_sales_renamed['date_sales'].dt.strftime('%Y-%m-%d')


#Create and format date_stock to yyyy-mm-dd according to last week sunday for stock dataframe
day_sunday = pd.to_datetime(nexsys_sales_renamed['date_sales'].iloc[0]) #Use the first row date_sales
end_week = day_sunday + timedelta(days=(6 - day_sunday.weekday()))
end_week_formatted = end_week.strftime('%Y-%m-%d')

#Create date_stock column
nexsys_stock_renamed['date_stock'] = end_week_formatted

Format Tecnoglobal dfs by column name

In [ ]:
#Each df must have the following columns:
#disti_sales: country_id, importer_name, date_sales, part_number, sku, invoice, reseller_code, sell_out_units, sell_out_value
#disti_stock: country_id, importer_name, date_stock, stock_units, stock_costs, part_number
 
# Convert all columns to lowercase, replace spaces with underscores and strip leading/trailing spaces.
input_tecnoglobal_sales.columns = input_tecnoglobal_sales.columns.str.strip().str.lower().str.replace(" ", "_")
input_tecnoglobal_stock.columns = input_tecnoglobal_stock.columns.str.strip().str.lower().str.replace(" ", "_")


#Define columns to rename
columns_to_rename_tecnoglobal_sales = {
                    'razon_social': 'reseller_origin', #resellers name for Tecnoglobal
                    'codigoarticulo': 'sku', #SKU for Tecnoglobal distributor
                    'cod_fabricante': 'part_number', #Part number from ASUS
                    'fechadocto' : 'date_sales', #Date sales reported by Tecnoglobal
                    'nro_docto': 'invoice', #Invoice number
                    'cantidad': 'sell_out_units', #Units sold
                    'total_us': 'sell_out_value' ,#Sales in USD
                    'rut': 'reseller_code' #Resellers code for Tecnoglobal
}
 
columns_to_rename_tecnoglobal_stock = {
    'codigoarticulo': 'sku', #SKU for Tecnoglobal distributor
    'cantidadexis': 'stock_units', #Stock in units
    'costounitario': 'stock_usd_units', #Stock in value (Tecnoglobal provides it)
    'partnumber': 'part_number' #part_number from ASUS
}
 
#Rename columns in dataframe
tecnoglobal_sales_renamed = input_tecnoglobal_sales.rename(columns = columns_to_rename_tecnoglobal_sales)
tecnoglobal_stock_renamed = input_tecnoglobal_stock.rename(columns = columns_to_rename_tecnoglobal_stock)
 
#set [stock_usd] = [stock_value] because report is in USD
tecnoglobal_stock_renamed['stock_value'] = tecnoglobal_stock_renamed['stock_usd_units'] * tecnoglobal_stock_renamed['stock_units']
 
#set [currency_code] = USD
tecnoglobal_sales_renamed['currency_code'] = 'USD'
tecnoglobal_stock_renamed['currency_code'] = 'USD'
 
#No reported date for tecnoglobal. Instead use monday of last week
date_week = dt.today().date()
#this_week_monday = date_week - timedelta(days=date_week.weekday())
#last_monday = this_week_monday - timedelta(days=7)
#tecnoglobal_sales_renamed['date_sales'] = last_monday
 
#Next lines are in case of reported dates in excel file. Ignore if not.
tecnoglobal_sales_renamed['date_sales'] = pd.to_datetime(tecnoglobal_sales_renamed['date_sales'], format='%d/%m/%Y', errors='coerce')
tecnoglobal_sales_renamed['date_sales'] = tecnoglobal_sales_renamed['date_sales'].dt.strftime('%Y-%m-%d')
 
# Drop columns in sales and stock dataframes
tecnoglobal_sales_renamed = tecnoglobal_sales_renamed[columns_to_maintain_sales]
tecnoglobal_stock_renamed = tecnoglobal_stock_renamed[columns_to_maintain_stock]
 
#Remove "." symbol from [reseller_code] and 0s from the string
tecnoglobal_sales_renamed['reseller_code'] = tecnoglobal_sales_renamed['reseller_code'].str.replace(r'[.]', '', regex=True)
tecnoglobal_sales_renamed['reseller_code'] = tecnoglobal_sales_renamed['reseller_code'].apply(lambda x: "-".join([x.split("-")[0].lstrip("0")] + x.split("-")[1:]))
 
#country_id = 1 is Chile code
tecnoglobal_sales_renamed['country_id'] = 1
tecnoglobal_stock_renamed['country_id'] = 1
 
#For Chile: importer_id = 5 for Intcomex, 13 for Ingram, 22 for Nexsys, 23 for Tecnoglobal
tecnoglobal_sales_renamed['importer_id'] = 23
tecnoglobal_stock_renamed['importer_id'] = 23
 
 
#Create and format date_stock to yyyy-mm-dd according to last week sunday for stock dataframe
# Check if the DataFrame is empty
if not tecnoglobal_sales_renamed.empty:
    # Use the first row's `date_sales`
    day_sunday = pd.to_datetime(tecnoglobal_sales_renamed['date_sales'].iloc[0])
else:
    # Get last week's Sunday
    today = pd.Timestamp.today()
    # If sales report is empty and not recent, modify this line to preferred week
    #day_sunday = pd.to_datetime("9/2/25", format="%d/%m/%y")
    day_sunday = today - pd.Timedelta(days=today.weekday() + 1)  # Move to last Sunday
 
end_week = day_sunday + timedelta(days=(6 - day_sunday.weekday()))
end_week_formatted = end_week.strftime('%Y-%m-%d')
 
#Create date_stock column
tecnoglobal_stock_renamed['date_stock'] = end_week_formatted

Format PCFactory dfs by column name

In [ ]:
#Each df must have the following columns:
#disti_sales: country_id, importer_name, date_sales, part_number, sku, invoice, reseller_code, reseller_name, sell_out_units, sell_out_value
#disti_stock: country_id, importer_name, date_stock, stock_units, stock_costs, part_number

# Convert all columns to lowercase, replace spaces with underscores and strip leading/trailing spaces.
input_pcfactory_sales.columns = input_pcfactory_sales.columns.str.strip().str.lower().str.replace(" ", "_")
input_pcfactory_stock.columns = input_pcfactory_stock.columns.str.strip().str.lower().str.replace(" ", "_")

#Define columns to rename
columns_to_rename_pcfactory_sales = {
                    'reseller': 'reseller_origin', #resellers name for PCFactory
                    'id_pcf': 'sku', #SKU for PCFactory distributor
                    'p#': 'part_number', #Part number from ASUS
                    'fecha' : 'date_sales', #Date sales reported by PCFactory
                    'invoice': 'invoice', #Invoice number
                    'cantidad': 'sell_out_units', #Units sold
                    'neto': 'sell_out_value' ,#Sales in CLP, use exrate table from DB
                    'tax_id': 'reseller_code' #Resellers code for  PCF
}

columns_to_rename_pcfactory_stock = {
    'id_pcf': 'sku', #SKU for PCF distributor
    'stock_pcf': 'stock_units', #Stock in units
    'purchase': 'stock_value', #Stock in value (PCF provides it) in USD
    'p#': 'part_number' #part_number from ASUS
}

# Rename columns in dataframe
pcfactory_sales_renamed = input_pcfactory_sales.rename(columns = columns_to_rename_pcfactory_sales)
pcfactory_stock_renamed = input_pcfactory_stock.rename(columns = columns_to_rename_pcfactory_stock)

#set [currency_code] = USD for stock and CLP for sales
pcfactory_sales_renamed['currency_code'] = 'CLP'
pcfactory_stock_renamed['currency_code'] = 'USD'

# Drop columns in sales and stock dataframes
pcfactory_sales_renamed = pcfactory_sales_renamed[columns_to_maintain_sales]
pcfactory_stock_renamed = pcfactory_stock_renamed[columns_to_maintain_stock]

#country_id = 1 is Chile code, 2 is Peru code
pcfactory_sales_renamed['country_id'] = 1
pcfactory_stock_renamed['country_id'] = 1

#For Chile: importer_id = 5 for Intcomex, 13 for Ingram, 22 for Nexsys, 2 for PCF
pcfactory_sales_renamed['importer_id'] = 2
pcfactory_stock_renamed['importer_id'] = 2

#Format reported is %Y-%m-%d %H:%M:%S. Format used is '%Y-%m-%d'
pcfactory_sales_renamed['date_sales'] = pd.to_datetime(pcfactory_sales_renamed['date_sales'], format='%Y-%m-%d %H:%M:%S', errors='coerce')

#Create and format date_stock to yyyy-mm-dd according to last week sunday for stock dataframe
day_sunday = pd.to_datetime(pcfactory_sales_renamed['date_sales'].max()) #Use the first row date_sales
end_week = day_sunday + timedelta(days=(6 - day_sunday.weekday()))
end_week_formatted = end_week.strftime('%Y-%m-%d')

#Create date_stock column
pcfactory_stock_renamed['date_stock'] = end_week_formatted

Concatenate all sales and stock dataframes

In [ ]:
#With all distis having the same column format and type, concatenate all to work with only 2 dataframes
distis_sales = pd.concat([ingram_sales_renamed, intcomex_sales_renamed, nexsys_sales_renamed, tecnoglobal_sales_renamed, pcfactory_sales_renamed], ignore_index= True)
distis_stock = pd.concat([ingram_stock_renamed, intcomex_stock_renamed, nexsys_stock_renamed, tecnoglobal_stock_renamed, pcfactory_stock_renamed], ignore_index= True)

distis_sales.to_csv(r'../output/01_sales_concatenated.csv', index=False)
distis_stock.to_csv(r'../output/01_stock_concatenated.csv', index=False)

Format sales and stock by dtype

In [ ]:
#Set New Dtypes
#str = string (text) / float = decimal number / int = integers / datetime = date
new_distis_sales_types = {
    'sku': str,
    'invoice': str,
    'reseller_code': str,
    'date_sales': 'datetime64[ns]',
    'sell_out_units': float,
    'sell_out_value': float,
    'currency_code': str,
    'part_number':str,
    'country_id': int,
    'importer_id':int,
    'reseller_origin':str
}

new_distis_stock_types = {
    'sku': str,
    'date_stock': 'datetime64[ns]',
    'stock_units': float,
    'stock_value': float,
    'currency_code': str,
    'importer_id': int,
    'country_id':int,
    'part_number':str
}

#Step 1: Set new datatypes to columns
distis_sales = distis_sales.astype(new_distis_sales_types)
distis_stock = distis_stock.astype(new_distis_stock_types)

#Step 2: Fill empty values with na
distis_sales.replace('', np.nan, inplace=True)
distis_stock.replace('', np.nan, inplace=True)

#Step 3: format everything to lower case to make it easier to work with
distis_sales_df_lower = distis_sales.map(lambda s:s.lower() if type(s) == str else s)
distis_stock_df_lower = distis_stock.map(lambda s:s.lower() if type(s) == str else s)

#### Format Columns

In [ ]:
#Create a copy of df to preserve last changes.
distis_sales_format = distis_sales_df_lower.copy()
distis_stock_format = distis_stock_df_lower.copy()

Functions that help format columns

In [ ]:
#Format sell_out_units and sell_out_value
# Step 1: Define function that will fix decimal formats
def correct_decimal_format(number_to_check):
    number_to_check = str(number_to_check)  # Ensure the value is treated as a string
    # Scenario 1: value has both , and .
    if ',' in number_to_check and '.' in number_to_check:
        #If , appears before . then:
        if number_to_check.rfind(',') < number_to_check.rfind('.'):
            # Last separator is '.', which means ',' is thousands separator
            return number_to_check.replace(',', '')
        else:
            # Last separator is ',', which means '.' is thousands separator and ',' is decimal separator
            return number_to_check.replace('.', '').replace(',', '.')
    elif ',' in number_to_check:
        # Scenario 2: Only ',' is present, likely as decimal separator
        return number_to_check.replace(',', '.')
    else:
        # Scenario 3: Only '.' is present or no separators at all, likely already in correct format
        return number_to_check

Convert to uppercase string columns

In [ ]:
#Apply Upper: this should be applied to, sku, invoice and reseller_code
columns_to_uppercase = ['sku', 'invoice', 'reseller_code', 'currency_code']

#Apply the uppercase transformation and delete spaces to each column in the list for each dataframe
for column in columns_to_uppercase:
    if column in distis_sales_format.columns:
        distis_sales_format[column] = distis_sales_format[column].astype(str).str.upper() #Uppercase transformation for sales
        distis_sales_format[column] = distis_sales_format[column].str.replace(r'\s+', '', regex=True) #Delete spaces from text for sales

    if column in distis_stock_format.columns:
        distis_stock_format[column] = distis_stock_format[column].astype(str).str.upper() #Uppercase transformation for stock
        distis_stock_format[column] = distis_stock_format[column].str.replace(r'\s+', '', regex=True) #Delete spaces from text for stock

#Apply uppercase to reseller_name column
distis_sales_format['reseller_origin'] = distis_sales_format['reseller_origin'].astype(str).str.upper()

#Apply uppercase to part_number column
distis_sales_format['part_number'] = distis_sales_format['part_number'].astype(str).str.upper()
distis_stock_format['part_number'] = distis_stock_format['part_number'].astype(str).str.upper()

#Delete final spaces from right side of a string for part_number
distis_sales_format['part_number'] = distis_sales_format['part_number'].str.rstrip()
distis_stock_format['part_number'] = distis_stock_format['part_number'].str.rstrip()

#Delete in-between spaces from part_numbers if they start with '90'
distis_sales_format.loc[distis_sales_format['part_number'].str.startswith('90'), 'part_number'] = distis_sales_format['part_number'].str.replace(r'\s+', '', regex=True)
distis_stock_format.loc[distis_stock_format['part_number'].str.startswith('90'), 'part_number'] = distis_stock_format['part_number'].str.replace(r'\s+', '', regex=True)

#Delete final spaces from right side of a string for reseller_origin
distis_sales_format['reseller_origin'] = distis_sales_format['reseller_origin'].str.rstrip()

Fix decimal formats and round float columns

In [ ]:
# Step 1: Define columns for data manipulation
columns_correct_decimal = ['sell_out_units', 'sell_out_value', 'sell_out_usd', 'stock_units', 'stock_value']

# Step 2: apply function that fixes decimal formats
for column in columns_correct_decimal:
    if column in distis_sales_format.columns:
        distis_sales_format[column] = distis_sales_format[column].apply(correct_decimal_format).astype(float) #apply function to fix format issues
    if column in distis_stock_format.columns:
        distis_stock_format[column] = distis_stock_format[column].apply(correct_decimal_format).astype(float) #apply function to fix format issues

# Step 3: apply function that fixes decimal formats and round to 0 for units and 2 for usd/costs
distis_sales_format['sell_out_units'] = round(distis_sales_format['sell_out_units'], 0) #round to 0 decimals
distis_sales_format['sell_out_value'] = round(distis_sales_format['sell_out_value'], 2) #round to 2 decimals
distis_stock_format['stock_units'] = round(distis_stock_format['stock_units'], 0) #round to 0 decimals
distis_stock_format['stock_value'] = round(distis_stock_format['stock_value'], 2) #round to 2 decimals

In [ ]:
#Format date_sales to yyyy-mm-dd format
#distis_sales_format['date_sales'] = distis_sales_format['date_sales'].dt.strftime('%Y-%m-%d')
#distis_stock_format['date_stock'] = distis_stock_format['date_stock'].dt.strftime('%Y-%m-%d')

#Define df_sanitized for upon updates in upload archive
df_sales_sanitized = distis_sales_format.copy()
df_stock_sanitized = distis_stock_format.copy()

df_sales_sanitized.to_csv(r'../output/01_df_sales_sanitized.csv', index=False)
df_stock_sanitized.to_csv(r'../output/01_df_stock_sanitized.csv', index=False)

#### Format [Nexsys, Ingram, Tecnoglobal] reseller_codes

Create reseller dataframe

In [ ]:
#These distis use the same [reseller_code] value, meaning that we can generate entries for both.
#They use the chilean RUT to identify them with some differences that can be formatted:

# Ingram: Rut with the unique identifier included. When its below 10 million, it uses a 0 at the start
# Nexsys: Rut without unique identifier.
# Tecnoglobal: Rut with the unique identifier included. It uses 0s at the start to maintain 10-length before "-" string 
# PCFactory: Rut with the unique identifier included.

#Create dataframe to analize resellers by name and code
df_reseller = df_sales_sanitized[['country_id', 'importer_id', 'reseller_code', 'reseller_origin']]
df_reseller = df_reseller.drop_duplicates(subset=['importer_id', 'reseller_code']) #dropping duplicates by code and importer_name

#Copy to maintain changes from df_reseller
reseller_formatted = df_reseller.copy()

In [ ]:
#####################################################
# 1) generate [national_id] and format it
# Since each disti works the reseller_codes in different formats, we need to standarize them first.
#####################################################

# Remove "." symbol
reseller_formatted['national_id'] = reseller_formatted['reseller_code'].str.replace(r'[.]', '', regex=True)

# Add a 0 at the start of the string if the length before the "-" is < 8
reseller_formatted['national_id'] = reseller_formatted['national_id'].apply(
    lambda x: x if len(x.split('-')[0]) >= 8 else x.split('-')[0].zfill(8) + ('-' + x.split('-')[1] if '-' in x else '')
)

# Add Rut's verification digit to Nexsys [national_id]
# a) Function to calculate the verification digit (DV) for a RUT
def calculate_verification_digit(rut_body):
    # Ensure only numeric characters are used
    rut_body = ''.join(filter(str.isdigit, rut_body))
    
    reversed_digits = map(int, reversed(rut_body))
    factors = cycle(range(2, 8))
    s = sum(d * f for d, f in zip(reversed_digits, factors))
    remainder = (-s) % 11
    return str(remainder) if remainder < 10 else 'K'

# b) Add verification digit to Nexsys only
reseller_formatted.loc[reseller_formatted['importer_id'] == 22, 'national_id'] = (
    reseller_formatted.loc[reseller_formatted['importer_id'] == 22, 'national_id']
    .apply(lambda x: f"{x}-{calculate_verification_digit(x)}")
)

In [ ]:
#####################################################
# 2) For each [reseller_code, importer_id] generates duplicates for other [importer_ids]
# We will create reseller_codes for each disti that uses RUT to identify their resellers.
# PCFactory won't have new resellers included in table. If other distis have a new reseller, then is created for PCFactory.
# New PCFactory resellers will be named PCF and be replaced with another reseller_code
#####################################################

# Define a dictionary to assign new importer ids based on current ones
importer_mapping = {
    13: [2, 22, 23],
    22: [2, 13, 23],
    23: [2, 22, 13]
}

# Generate a copy of the df and assign new importer_id values
reseller_duplicates = reseller_formatted.copy()
reseller_duplicates['importer_format_to_convert'] = reseller_duplicates['importer_id'].map(importer_mapping)

# Explode the list values into separate rows
reseller_duplicates = reseller_duplicates.explode('importer_format_to_convert', ignore_index=True)

# Concatenate original and duplicated DataFrames
reseller_formatted = pd.concat([reseller_formatted, reseller_duplicates], ignore_index=True)

In [ ]:
#####################################################
# 3) Generate [new_reseller_code] based on the value of [importer_format_to_convert]
# We will create reseller_codes for each disti that uses RUT to identify their resellers.
#####################################################

reseller_formatted['new_reseller_code'] = reseller_formatted['national_id']

# For [importer_format_to_convert] = 13 or 23, add 0 at the start if len < 8 before the "-"
reseller_formatted.loc[reseller_formatted['importer_format_to_convert'].isin([2, 13, 23]), 'new_reseller_code'] = (
    reseller_formatted.loc[reseller_formatted['importer_format_to_convert'].isin([2, 13, 23]), 'new_reseller_code']
    .apply(lambda x: x if len(x.split('-')[0]) >= 8 else x.split('-')[0].zfill(8) + ('-' + x.split('-')[1] if '-' in x else ''))
)

# For [importer_format_to_convert] = 22, remove leading 0 and the last 2 characters
reseller_formatted.loc[reseller_formatted['importer_format_to_convert'] == 22, 'new_reseller_code'] = (
    reseller_formatted.loc[reseller_formatted['importer_format_to_convert'] == 22, 'new_reseller_code']
    .apply(lambda x: x[:-2].lstrip('0'))
)

#####################################################
# 4) Update [importer_id, reseller_code] where importer_format_to_convert != NaN
#####################################################

# Set dtype = Int64 (Int64 is <> to int64 because it supports NaN values)
reseller_formatted['importer_format_to_convert'] = reseller_formatted['importer_format_to_convert'].astype('Int64')

# Set a mask to filter df and update corresponding rows
mask = reseller_formatted['importer_format_to_convert'] != 0
reseller_formatted.loc[mask, 'importer_id'] = reseller_formatted.loc[mask, 'importer_format_to_convert']
reseller_formatted.loc[mask, 'reseller_code'] = reseller_formatted.loc[mask, 'new_reseller_code']


# Keep only necessary columns
reseller_formatted = reseller_formatted[['country_id', 'importer_id', 'reseller_code', 'reseller_origin', 'national_id']]

#### Format [Nexsys, Ingram, Tecnoglobal, PCFactory] Reseller Names

In [ ]:
# Create empty columns that will be uploaded to reseller_list table in SQL
reseller_formatted['reseller_name'] = None
reseller_formatted['country_segment'] = None
reseller_formatted['asus_segment'] = None
reseller_formatted['country_id'] = 1

# Define function that process [reseller_origin] and assigns value to [reseller_name]
def format_reseller_name(df, column_name):
    # Step 1: Identify rows where [national_id] is from an enterprise and is not Intcomex
    # We exclude intcomex because they don't provide information about their resellers.
    mask = reseller_formatted['importer_id'] != 5

    # Step 2: Set `reseller_name` to the value of `reseller_origin` for matching rows
    df.loc[mask, column_name] = df.loc[mask, 'reseller_origin']

    # Step 3: Convert `reseller_name` to uppercase
    df.loc[mask, column_name] = df.loc[mask, column_name].str.upper()

    # Step 4: Remove all symbols from `reseller_name`
    symbols_to_remove = ['.', '"', ',', ';', ':', "'", '?', '!']
    pattern = f"[{''.join(map(re.escape, symbols_to_remove))}]"
    df.loc[mask, column_name] = df.loc[mask, column_name].str.replace(pattern, '', regex=True)

    # Step 5: Normalize text by removing accents
    df[column_name] = df[column_name].apply(
        lambda x: unicodedata.normalize('NFKD', x).encode('ascii', 'ignore').decode('utf-8') if isinstance(x, str) else x
    )

    # Step 6: Remove specific unwanted words
    words_to_remove = ['SAC', 'EIRL', 'SA', 'LTDA', 'SOCIEDAD', 'ANONIMA', 'SPA', 'RESPONSABILIDAD', 'LTD',
                       'ABIERTA', 'CERRADA', 'LIMITADA', 'SRL', 'SCR', 'EIR', 'SCRL', 'LTD', 'COMERCIAL', 'SOC', 'COM']
    df.loc[mask, column_name] = (
        df.loc[mask, column_name]
        .str.split()  # Split into words
        .apply(lambda words: ' '.join([word for word in words if word not in words_to_remove]))  # Remove unwanted words
    )

    # Step 7: Clean up any remaining extra spaces
    df.loc[mask, column_name] = df.loc[mask, column_name].str.strip().str.replace(r'\s+', ' ', regex=True)

    # Step 8: Convert `reseller_name` to title case
    df.loc[mask, column_name] = df.loc[mask, column_name].str.title()

    return df

# Apply Function to [reseller_name]
reseller_formatted = format_reseller_name(reseller_formatted, 'reseller_name')

In [ ]:
# Extract the numeric part before the '-' symbol and convert to numeric.
reseller_formatted['national_id_numeric'] = reseller_formatted['national_id'].astype(str).str.split('-').str[0]
reseller_formatted['national_id_numeric'] = pd.to_numeric(reseller_formatted['national_id_numeric'], errors='coerce')

# Create to filter only natural persons RUT
natural_person_mask = (reseller_formatted['national_id_numeric'] < 40000000) & (reseller_formatted['importer_id'] != 5)

# Assign values based on the mask
reseller_formatted['country_segment'] = np.where(natural_person_mask, 'Soho', 'Pending')
reseller_formatted['asus_segment'] = np.where(natural_person_mask, 'Smb', 'Pending')

# Drop helper column if no longer needed
reseller_formatted.drop(columns=['national_id_numeric'], inplace=True)

reseller_formatted.to_csv(r'../output/01_resellers_created.csv', index=False)